In [2]:
import pandas as pd
import difflib

def definite_match(yukka_name, stoxx_names):
    """Function for definite matching based on exact string match."""
    return stoxx_names.get(yukka_name)

def rough_match(yukka_name, stoxx_names):
    """Function for rough matching based on partial string similarity."""
    matches = difflib.get_close_matches(yukka_name, stoxx_names.keys(), n=1, cutoff=0.6)
    return matches[0] if matches else None

# Load the Yukka Stoxx600 CSV file
yukka_stoxx600_df = pd.read_csv('/Users/maxbreitruck/Downloads/Yukka_Stoxx600.csv')

# Load the STOXX Europe 600 Excel file
stoxx600_excel_df = pd.read_excel('/Users/maxbreitruck/Downloads/SXXGR.xlsx')

# Normalize company names for matching and remove duplicates
stoxx600_excel_df['Normalized Company Name'] = stoxx600_excel_df['Company'].str.lower()
stoxx600_excel_df = stoxx600_excel_df.drop_duplicates(subset='Normalized Company Name')

# Create dictionaries for definite and rough matching
stoxx600_definite_dict = stoxx600_excel_df.set_index('Normalized Company Name').to_dict('index')
stoxx600_rough_dict = {row['Normalized Company Name']: row for _, row in stoxx600_excel_df.iterrows()}

# Normalize company names in Yukka dataframe for matching
yukka_stoxx600_df['Normalized Company Name'] = yukka_stoxx600_df['Subfolder Name'].apply(lambda x: x.split('_')[0].lower())

# Perform the definite and rough matching
yukka_stoxx600_df['Definite Matched Company Name'] = yukka_stoxx600_df['Normalized Company Name'].apply(
    lambda name: definite_match(name, stoxx600_definite_dict)
)
yukka_stoxx600_df['Rough Matched Company Name'] = yukka_stoxx600_df['Normalized Company Name'].apply(
    lambda name: rough_match(name, stoxx600_rough_dict)
)

# Retrieve the matched sector and company name for both matchings
yukka_stoxx600_df['Definite Matched Supersector'] = yukka_stoxx600_df['Definite Matched Company Name'].apply(
    lambda name: stoxx600_definite_dict.get(name, {}).get('Supersector')
)
yukka_stoxx600_df['Rough Matched Supersector'] = yukka_stoxx600_df['Rough Matched Company Name'].apply(
    lambda name: stoxx600_rough_dict.get(name, {}).get('Supersector')
)

# Drop unnecessary columns
yukka_stoxx600_df.drop(columns=['Normalized Company Name', 'Definite Matched Company Name', 'Rough Matched Company Name'], inplace=True)

# Save the updated DataFrame to a new CSV file
yukka_stoxx600_df.to_csv('updated_yukka_stoxx600_with_both_matchings.csv', index=False)

print("Matching completed and file saved as 'updated_yukka_stoxx600_with_both_matchings.csv'")


TypeError: unhashable type: 'dict'